# 04 — Face Model Prototyping

Prototype the facial expression classification branch:
- ResNet-18 backbone
- Grad-CAM visualization
- Training experiments
- Emotion → depression mapping

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from ml.face_pipeline.cnn_model import FaceCNN
from ml.face_pipeline.preprocessing import preprocess_image_bytes
from ml.face_pipeline.vision_branch import generate_gradcam

# Initialize model
model = FaceCNN(num_classes=2)
print(f'Embed dim: {model.embed_dim}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Test with dummy image
import numpy as np
from PIL import Image
import io

img = Image.new('RGB', (224, 224), color=(128, 128, 128))
buf = io.BytesIO()
img.save(buf, format='PNG')
img_bytes = buf.getvalue()

tensor = preprocess_image_bytes(img_bytes)

with torch.no_grad():
    logits, embedding = model(tensor)

print(f'Logits: {logits.shape} → {logits}')
print(f'Embedding: {embedding.shape}')

## Grad-CAM Visualization

See which parts of the face the model focuses on for depression detection.

In [ ]:
from ml.face_pipeline.vision_branch import VisionBranch

vision_model = VisionBranch()
heatmap = generate_gradcam(vision_model, tensor)

print(f'Heatmap shape: {heatmap.shape}')
print(f'Heatmap range: [{heatmap.min():.3f}, {heatmap.max():.3f}]')

import matplotlib.pyplot as plt
plt.imshow(heatmap, cmap='jet')
plt.colorbar()
plt.title('Grad-CAM Heatmap')
plt.show()

## Next Steps

- Run full training with `src/face_pipeline/train_face_model.py`
- Experiment with backbone freezing strategies
- Test on real facial expression images